# NF-MCMC Ackley Demo

This notebook checks the PyTorch `NFMCMCND` implementation on a 2D Ackley target. It compares:

- random-walk MCMC samples as a truth/reference cloud,
- raw normalizing-flow samples,
- normalizing-flow proposals after Metropolis-Hastings correction.

The implementation is conceptually the same correction idea as Yaohang's NF-guided sampling slides, but uses the project PyTorch RealNVP-based sampler instead of TensorFlow Probability MAF.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in [p, *p.parents]:
        if (candidate / "src" / "quantom_ips").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing src/quantom_ips")

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from quantom_ips.envs.samplers.nf_mcmc_nd import NFMCMCND

torch.manual_seed(7)
np.random.seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print("repo:", REPO_ROOT)
print("device:", device)

## Ackley target

The old notebook used an inverted Ackley weight. Here we use the same style: high probability where `ackley_max - ackley(x, y)` is large.

In [ ]:
R_MIN, R_MAX = -2.0, 2.0
EPS = 1e-8

def ackley_np(x, y):
    return -20.0 * np.exp(-0.2 * np.sqrt(0.5 * (x**2 + y**2))) - np.exp(
        0.5 * (np.cos(2 * np.pi * x) + np.cos(2 * np.pi * y))
    ) + np.e + 20.0

fine = np.linspace(R_MIN, R_MAX, 800)
xx_fine, yy_fine = np.meshgrid(fine, fine, indexing="ij")
ACKLEY_MAX = float(ackley_np(xx_fine, yy_fine).max())

def target_weight_np(points):
    points = np.asarray(points)
    x = points[..., 0]
    y = points[..., 1]
    inside = (x >= R_MIN) & (x <= R_MAX) & (y >= R_MIN) & (y <= R_MAX)
    weight = np.maximum(ACKLEY_MAX - ackley_np(x, y), EPS)
    return np.where(inside, weight, 0.0)

def ackley_grid_density(grid_n=80):
    x_axis = torch.linspace(R_MIN, R_MAX, grid_n, device=device, dtype=dtype)
    y_axis = torch.linspace(R_MIN, R_MAX, grid_n, device=device, dtype=dtype)
    xx, yy = torch.meshgrid(x_axis, y_axis, indexing="ij")
    z = -20.0 * torch.exp(-0.2 * torch.sqrt(0.5 * (xx**2 + yy**2))) - torch.exp(
        0.5 * (torch.cos(2 * torch.pi * xx) + torch.cos(2 * torch.pi * yy))
    ) + torch.e + 20.0
    density = torch.clamp(torch.tensor(ACKLEY_MAX, device=device, dtype=dtype) - z, min=EPS)
    return density, [x_axis, y_axis]

density, axes = ackley_grid_density(grid_n=80)
print("density shape:", tuple(density.shape), "min/max:", float(density.min()), float(density.max()))

## Reference random-walk MCMC

This is just for visual reference in the notebook. The project sampler we are checking is the NF-MCMC correction below.

In [ ]:
def random_walk_mcmc(n_samples=60_000, burn_in=5_000, step_size=0.16, start=(0.0, 0.0)):
    total = n_samples + burn_in
    states = np.zeros((n_samples, 2), dtype=np.float64)
    current = np.asarray(start, dtype=np.float64)
    current_w = float(target_weight_np(current))
    accepted = 0
    out_i = 0
    for i in range(total):
        proposal = current + np.random.normal(0.0, step_size, size=2)
        proposal_w = float(target_weight_np(proposal))
        ratio = proposal_w / max(current_w, EPS)
        if np.random.rand() < min(1.0, ratio):
            current = proposal
            current_w = proposal_w
            accepted += 1
        if i >= burn_in:
            states[out_i] = current
            out_i += 1
    return states, accepted / total

truth_samples, truth_acc = random_walk_mcmc()
print(f"reference random-walk MCMC acceptance = {truth_acc:.3f}")

## Train NF proposal and apply MH correction

`raw_nf_samples` show the learned proposal distribution before correction. `nf_mcmc_samples` are the corrected samples. If the NF proposal is well trained, the correction acceptance rate should be high.

In [ ]:
N_EVENTS = 60_000
NF_TRAIN_STEPS = 1000
NF_TRAIN_SAMPLES = 25_000

sampler = NFMCMCND(
    train_steps=NF_TRAIN_STEPS,
    train_samples=NF_TRAIN_SAMPLES,
    batch_size=1024,
    flow_layers=6,
    hidden_dim=128,
    burn_in=20,
    thin=5,
)

cache_key = ("ackley-2d-demo", NF_TRAIN_STEPS, NF_TRAIN_SAMPLES)
flow = sampler._get_flow(density, axes, cache_key=cache_key)

with torch.no_grad():
    raw_unit, _ = flow.sample(N_EVENTS, device=device, dtype=dtype)
    raw_nf_samples = sampler._from_unit(raw_unit, axes).detach().cpu().numpy()

nf_mcmc_t, nf_mcmc_acc = sampler.sample(density, axes, N_EVENTS, cache_key=cache_key)
nf_mcmc_samples = nf_mcmc_t.detach().cpu().numpy()
print(f"NF-MCMC correction acceptance = {nf_mcmc_acc:.3f}")

## 2D distributions

This mirrors the slide: truth/reference MCMC, NF-MCMC correction, and raw NF proposal.

In [ ]:
def plot_2d_comparison(truth, corrected, raw_nf, bins=120):
    fig, axes_plot = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
    panels = [
        ("MCMC reference", truth),
        (f"NF-MCMC correction\nacceptance={nf_mcmc_acc:.3f}", corrected),
        ("Raw NF proposal", raw_nf),
    ]
    for ax, (title, samples) in zip(axes_plot, panels):
        ax.hist2d(samples[:, 0], samples[:, 1], bins=bins, range=[[R_MIN, R_MAX], [R_MIN, R_MAX]], cmap="viridis")
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_xlim(R_MIN, R_MAX)
        ax.set_ylim(R_MIN, R_MAX)
    return fig

plot_2d_comparison(truth_samples, nf_mcmc_samples, raw_nf_samples)
plt.show()

## Marginal checks

Top row: truth/reference MCMC vs NF-MCMC correction. Bottom row: truth/reference MCMC vs raw NF.

In [ ]:
def plot_marginals(truth, corrected, raw_nf, bins=80):
    fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
    labels = ["x", "y"]
    for dim in range(2):
        axs[0, dim].hist(truth[:, dim], bins=bins, range=(R_MIN, R_MAX), histtype="step", label="MCMC reference", color="tab:blue", linewidth=1.8)
        axs[0, dim].hist(corrected[:, dim], bins=bins, range=(R_MIN, R_MAX), histtype="step", label="NF-MCMC correction", color="tab:orange", linewidth=1.5)
        axs[0, dim].set_title(f"Correction vs truth - {labels[dim]} marginal")
        axs[0, dim].legend()

        axs[1, dim].hist(truth[:, dim], bins=bins, range=(R_MIN, R_MAX), histtype="step", label="MCMC reference", color="tab:blue", linewidth=1.8)
        axs[1, dim].hist(raw_nf[:, dim], bins=bins, range=(R_MIN, R_MAX), histtype="step", label="Raw NF", color="tab:orange", linewidth=1.5)
        axs[1, dim].set_title(f"Raw NF vs truth - {labels[dim]} marginal")
        axs[1, dim].legend()
    return fig

plot_marginals(truth_samples, nf_mcmc_samples, raw_nf_samples)
plt.show()

## Optional: acceptance rate vs dimensionality

This is slower. Set `RUN_DIMENSION_SWEEP = True` to run it. Keep `GRID_N_DIM` small for high dimensions because the target grid size grows as `GRID_N_DIM ** D`.

In [ ]:
RUN_DIMENSION_SWEEP = False
DIMS_TO_RUN = list(range(2, 8))
GRID_N_DIM = 7

def ackley_nd_grid(dim, grid_n=7):
    axes_nd = [torch.linspace(R_MIN, R_MAX, grid_n, device=device, dtype=dtype) for _ in range(dim)]
    grids = torch.meshgrid(*axes_nd, indexing="ij")
    r2 = sum(g**2 for g in grids) / dim
    c = sum(torch.cos(2 * torch.pi * g) for g in grids) / dim
    z = -20.0 * torch.exp(-0.2 * torch.sqrt(r2)) - torch.exp(c) + torch.e + 20.0
    density_nd = torch.clamp(z.max() - z, min=EPS)
    return density_nd, axes_nd

if RUN_DIMENSION_SWEEP:
    rates = []
    for dim in DIMS_TO_RUN:
        d, ax = ackley_nd_grid(dim, GRID_N_DIM)
        s = NFMCMCND(train_steps=250, train_samples=4096, batch_size=512, burn_in=10, thin=3, hidden_dim=128, flow_layers=6)
        _samples, rate = s.sample(d, ax, 3000, cache_key=("dim-sweep", dim))
        rates.append(rate)
        print(f"D={dim}: acceptance={rate:.3f}, grid={GRID_N_DIM}^{dim}")

    plt.figure(figsize=(6, 4))
    plt.plot(DIMS_TO_RUN, rates, marker="o")
    plt.xlabel("Dimensionality (D)")
    plt.ylabel("Acceptance rate")
    plt.title("NF-MCMC acceptance rate vs dimensionality")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("Dimension sweep skipped. Set RUN_DIMENSION_SWEEP = True to run it.")